# Week 84, Day 3: Evaluating models

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w084d3.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
import numpy as np

faulty = np.array([0] * 90 + [1] * 10)        # 1 means the part is faulty
pass_everything = np.zeros(100, dtype=int)
print("accuracy:", (pass_everything == faulty).mean())
print("faults caught:", int(((pass_everything == 1) & (faulty == 1)).sum()), "of", int(faulty.sum()))

In [ ]:
import numpy as np
from sklearn.metrics import confusion_matrix

y    = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1])
pred = np.array([0, 0, 0, 1, 0, 0, 1, 0, 1, 0])
tp = int(((y == 1) & (pred == 1)).sum())
fp = int(((y == 0) & (pred == 1)).sum())
fn = int(((y == 1) & (pred == 0)).sum())
tn = int(((y == 0) & (pred == 0)).sum())
print("TP", tp, "FP", fp, "FN", fn, "TN", tn)
print(confusion_matrix(y, pred).tolist())
print("accuracy", (tp + tn) / len(y))

In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

y    = [0, 0, 0, 0, 0, 0, 1, 1, 1, 1]
pred = [0, 0, 0, 1, 0, 0, 1, 0, 1, 0]
print(precision_score(y, pred), recall_score(y, pred), round(f1_score(y, pred), 3))
print(recall_score(y, [0] * 10), precision_score(y, [0] * 10, zero_division=0))

In [ ]:
import numpy as np
from sklearn.metrics import roc_auc_score

y      = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1])
scores = np.array([0.1, 0.3, 0.2, 0.6, 0.15, 0.4, 0.9, 0.35, 0.7, 0.45])
for t in (0.5, 0.4, 0.3):
    pred = scores >= t
    tp, fp = int((pred & (y == 1)).sum()), int((pred & (y == 0)).sum())
    print(t, "recall", tp / 4, "false positive rate", round(fp / 6, 2))
pairs = [(p, n) for p in scores[y == 1] for n in scores[y == 0]]
print(sum(p > n for p, n in pairs) / len(pairs), roc_auc_score(y, scores))

In [ ]:
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

y    = np.array([1000, 1150, 1450, 900, 1700])
pred = np.array([1020, 1100, 1400, 960, 1450])
print(mean_absolute_error(y, pred), round(float(np.sqrt(mean_squared_error(y, pred))), 1), round(r2_score(y, pred), 3))
residuals = y - pred
print(residuals.tolist(), int(np.abs(residuals).argmax()))

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier

X, y = load_iris(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
scores = cross_val_score(KNeighborsClassifier(5), X_train, y_train, cv=StratifiedKFold(5))
print(scores.round(3).tolist(), round(scores.mean(), 3), round(scores.std(), 3))
search = GridSearchCV(KNeighborsClassifier(), {"n_neighbors": [1, 3, 5, 7, 9, 15]}, cv=StratifiedKFold(5))
search.fit(X_train, y_train)
print(search.best_params_, round(search.best_score_, 3))
print("final test score:", round(search.score(X_test, y_test), 3))

> **Key idea.** On imbalanced data accuracy misleads; compare with the majority baseline and look at the confusion matrix (TP, FP, FN, TN; scikit-learn gives `[[TN, FP], [FN, TP]]`). Precision is TP/(TP+FP), the share of flags that were right; recall is TP/(TP+FN), the share of positives caught; F1 combines them. Choose between them by the cost of each mistake. Classifiers score, then threshold: lowering the threshold raises recall and usually lowers precision; choose it from costs on training or validation data. ROC AUC is the chance a positive outscores a negative (0.5 is random, 1 perfect). For regression report MAE, RMSE (weighs big errors) and R², with a baseline, and inspect the largest residuals. Cross-validate on training data (`cross_val_score`, stratified folds, preprocessing inside a pipeline) and tune with `GridSearchCV`; use the test set once, at the end.

### Exercise: The four boxes

Write `confusion(y, pred)`: a dict with `tp`, `fp`, `fn` and `tn` as whole numbers, where `1` is the positive class. Then `precision(c)`, `recall(c)` and `f1(c)` for such a dict, each rounded to 3 places, and `0.0` when the formula would divide by zero.

In [ ]:
import numpy as np

def confusion(y, pred):
    ___

def precision(c):
    ___

def recall(c):
    ___

def f1(c):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\nfrom sklearn.metrics import f1_score, precision_score, recall_score\n\ndef test_confusion():\n    \"from scratch, matching scikit-learn\"\n    y = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1])\n    pred = np.array([0, 0, 0, 1, 0, 0, 1, 0, 1, 0])\n    c = confusion(y, pred)\n    assert c == {\"tp\": 2, \"fp\": 1, \"fn\": 2, \"tn\": 5}\n    assert precision(c) == round(precision_score(y, pred), 3) == 0.667\n    assert recall(c) == 0.5 and f1(c) == round(f1_score(y, pred), 3)\n    none = confusion(y, np.zeros(10, dtype=int))\n    assert none == {\"tp\": 0, \"fp\": 0, \"fn\": 4, \"tn\": 6} and precision(none) == 0.0 and recall(none) == 0.0 and f1(none) == 0.0")

### Exercise: Moving the threshold

Write `sweep(y, scores, thresholds)`: for each threshold (a score at or above it counts as positive), a tuple `(threshold, precision, recall)` with both rounded to 3 places (`0.0` when undefined). Then `threshold_for_recall(y, scores, target, thresholds)`: the **highest** threshold whose recall is at least `target` (`None` if none is), and `cheapest(y, scores, thresholds, cost_fp, cost_fn)`: the threshold with the lowest total cost `FP × cost_fp + FN × cost_fn` (ties: the highest threshold).

In [ ]:
import numpy as np

def sweep(y, scores, thresholds):
    ___

def threshold_for_recall(y, scores, target, thresholds):
    ___

def cheapest(y, scores, thresholds, cost_fp, cost_fn):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\nY = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1])\nS = np.array([0.1, 0.3, 0.2, 0.6, 0.15, 0.4, 0.9, 0.35, 0.7, 0.45])\nT = [0.6, 0.5, 0.4, 0.3, 0.2]\n\ndef test_sweep():\n    \"lower thresholds catch more\"\n    assert sweep(Y, S, T) == [(0.6, 0.667, 0.5), (0.5, 0.667, 0.5), (0.4, 0.6, 0.75), (0.3, 0.571, 1.0), (0.2, 0.5, 1.0)]\n    assert threshold_for_recall(Y, S, 0.75, T) == 0.4 and threshold_for_recall(Y, S, 1.0, T) == 0.3\n    assert threshold_for_recall(Y, S, 1.0, [0.95]) is None\n    assert cheapest(Y, S, T, cost_fp=1, cost_fn=10) == 0.3\n    assert cheapest(Y, S, T, cost_fp=10, cost_fn=1) == 0.6")

### Exercise: ROC AUC by counting

Write `auc(y, scores)` from its meaning: over every pair of one positive and one negative example, the fraction where the positive has the higher score, counting a tie as half; rounded to 3 places. (It should match `roc_auc_score`.) Then `random_auc(y, rng)`: the AUC of random scores from `rng.random(len(y))`, which should be near 0.5.

In [ ]:
import numpy as np

def auc(y, scores):
    ___

def random_auc(y, rng):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\nfrom sklearn.metrics import roc_auc_score\n\ndef test_auc():\n    \"pairs of positive and negative\"\n    y = np.array([0, 0, 0, 0, 0, 0, 1, 1, 1, 1])\n    s = np.array([0.1, 0.3, 0.2, 0.6, 0.15, 0.4, 0.9, 0.35, 0.7, 0.45])\n    assert auc(y, s) == round(roc_auc_score(y, s), 3) == 0.875\n    assert auc([0, 1, 0, 1], [0.5, 0.5, 0.2, 0.9]) == 0.875, \"ties count half\"\n    assert auc([0, 1], [0.9, 0.1]) == 0.0\n    big = np.array([0, 1] * 500)\n    assert abs(random_auc(big, np.random.default_rng(0)) - 0.5) < 0.05")

### Exercise: Regression report

Write `regression_report(y, pred, baseline)`: a dict with `mae`, `rmse` and `r2` of the predictions, each rounded to 2 places, plus `baseline_mae`, the MAE of predicting the single number `baseline` for every example. Then `worst(y, pred, n)`: the positions of the `n` largest absolute residuals, largest first.

In [ ]:
import numpy as np

def regression_report(y, pred, baseline):
    ___

def worst(y, pred, n):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\n\ndef test_regression():\n    \"errors and residuals\"\n    y = np.array([1000, 1150, 1450, 900, 1700])\n    pred = np.array([1020, 1100, 1400, 960, 1450])\n    assert regression_report(y, pred, 1200) == {\"mae\": 86.0, \"rmse\": 119.58, \"r2\": 0.84, \"baseline_mae\": 260.0}\n    assert worst(y, pred, 2) == [4, 3]")

### Exercise: Cross-validation from scratch

Write `kfold_indices(n, k)`: split the positions `0` to `n − 1`, in order, into `k` consecutive folds whose sizes differ by at most one, with the larger folds first (like scikit-learn's `KFold` without shuffling); return a list of `(train, test)` pairs of lists. Then `cross_val(make_model, X, y, k)`: for each fold, fit a **new** model from `make_model()` on the training rows and score it on the test rows; return the list of scores, rounded to 3 places.

In [ ]:
import numpy as np

def kfold_indices(n, k):
    ___

def cross_val(make_model, X, y, k):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\nfrom sklearn.datasets import make_classification\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.model_selection import KFold, cross_val_score\n\ndef test_folds():\n    \"every example tested once\"\n    folds = kfold_indices(10, 3)\n    assert [test for _, test in folds] == [[0, 1, 2, 3], [4, 5, 6], [7, 8, 9]]\n    assert folds[1][0] == [0, 1, 2, 3, 7, 8, 9]\n    assert sorted(i for _, t in kfold_indices(23, 5) for i in t) == list(range(23))\n\ndef test_cross_val():\n    \"matches scikit-learn\"\n    X, y = make_classification(n_samples=120, n_features=4, random_state=3)\n    mine = cross_val(lambda: LogisticRegression(max_iter=1000), X, y, 4)\n    theirs = cross_val_score(LogisticRegression(max_iter=1000), X, y, cv=KFold(4)).round(3).tolist()\n    assert mine == theirs and len(mine) == 4")

### Exercise: Tuning without peeking

Write `tune_knn(X_train, y_train, ks)`: a `GridSearchCV` over `n_neighbors` in `ks`, with 5 stratified folds (`StratifiedKFold(5)`), wrapped around a pipeline that scales and then classifies (`make_pipeline(StandardScaler(), KNeighborsClassifier())`, whose parameter is called `kneighborsclassifier__n_neighbors`); fit it on the training data and return it. Then `final_score(search, X_test, y_test)`: the fitted search's test accuracy, rounded to 3 places.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def tune_knn(X_train, y_train, ks):
    ___

def final_score(search, X_test, y_test):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("from sklearn.datasets import load_iris\nfrom sklearn.model_selection import train_test_split\n\ndef test_tuning():\n    \"cross-validated on training data, tested once\"\n    X, y = load_iris(return_X_y=True)\n    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)\n    search = tune_knn(X_train, y_train, [1, 3, 5, 7, 9, 15])\n    best = search.best_params_[\"kneighborsclassifier__n_neighbors\"]\n    assert best in [1, 3, 5, 7, 9, 15] and search.best_score_ > 0.9\n    assert len(search.cv_results_[\"mean_test_score\"]) == 6\n    assert search.n_splits_ == 5\n    assert final_score(search, X_test, y_test) >= 0.89")

### Bug hunt: Ninety per cent of nothing

The report is meant to say how many faulty parts the model catches, but it prints accuracy, which looks fine: `0.9`. Report the **recall** of the faulty class instead, so it prints `recall: 0.0`.

In [ ]:
import numpy as np

faulty = np.array([0] * 9 + [1])
predicted = np.zeros(10, dtype=int)
print("recall:", float((predicted == faulty).mean()))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"the number accuracy hides\"\n    expect_output(\"recall: 0.0\")")

### Bug hunt: Precision with the wrong bottom

Precision divides by everything the model **flagged** (TP + FP), but this divides by everything that was really positive, which is recall: it prints `precision: 0.5`. Fix it so it prints `precision: 0.667`.

In [ ]:
tp, fp, fn = 2, 1, 2
precision = tp / (tp + fn)
print("precision:", round(precision, 3))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"TP over flagged\"\n    expect_output(\"precision: 0.667\")")

### Bug hunt: An RMSE that grows with the data

RMSE is the square root of the **mean** squared error, but this takes the square root of the **sum**, so the more examples there are the bigger it gets: `rmse: 267.39`. Fix it so it prints `rmse: 119.58`.

In [ ]:
import numpy as np

y = np.array([1000, 1150, 1450, 900, 1700])
pred = np.array([1020, 1100, 1400, 960, 1450])
print("rmse:", round(float(np.sqrt(((y - pred) ** 2).sum())), 2))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"mean, then square root\"\n    expect_output(\"rmse: 119.58\")")

### Puzzle: Tune, then test once

Arrange the lines so that hyperparameters are chosen by cross-validation on the training data and the test set is used exactly once, at the end. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
p = ___   # precision for TP=8, FP=2, FN=8
q = ___   # recall for the same
r = ___   # "precision" or "recall": what to favour when a miss is far costlier than a false alarm
s = ___   # the ROC AUC of scores that rank every positive above every negative
t = ___   # how many models GridSearchCV fits for 4 settings and cv=5, before the final refit

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == 0.8 and q == 0.5 and r == \"recall\", \"p, q and r\"\n    assert s == 1.0 and t == 20, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: a model report card

Write `report_card(model, X_train, y_train, X_test, y_test, cost_fp=1, cost_fn=10, thresholds=None)` for a binary classifier with `predict_proba` (positive class 1). `thresholds` defaults to `0.05, 0.10, ..., 0.95` (rounded to 2 places). Without touching the test set, choose the **tuned threshold**: get out-of-fold probabilities for the training data with `cross_val_predict(clone(model), X_train, y_train, cv=StratifiedKFold(5), method="predict_proba")`, and pick the threshold with the lowest cost `FP × cost_fp + FN × cost_fn` on them (ties: the highest threshold). Also cross-validate accuracy on the training data with the same folds. Then fit a fresh `clone(model)` on all the training data and report on the test set. Return a dict: `accuracy` (at 0.5), `baseline_accuracy` (always predicting the majority **training** class), `auc` (all 3 places), `cv_accuracy` (`(mean, std)`, 3 places), and `default` and `tuned`: for thresholds 0.5 and the tuned one, a dict with `threshold`, `confusion` (`tp`, `fp`, `fn`, `tn`), `precision`, `recall` (3 places, 0.0 when undefined) and `cost`.

In [ ]:
import numpy as np
from sklearn.base import clone
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict, cross_val_score

def report_card(model, X_train, y_train, X_test, y_test, cost_fp=1, cost_fn=10, thresholds=None):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import numpy as np\nfrom sklearn.datasets import make_classification\nfrom sklearn.linear_model import LogisticRegression\nfrom sklearn.model_selection import train_test_split\n\ndef _data():\n    X, y = make_classification(n_samples=600, n_features=5, n_informative=3, weights=[0.9], flip_y=0.02, random_state=7)\n    return train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)\n\ndef test_card():\n    \"accuracy, baseline, AUC and cross-validation\"\n    X_train, X_test, y_train, y_test = _data()\n    card = report_card(LogisticRegression(max_iter=1000), X_train, y_train, X_test, y_test)\n    assert card[\"baseline_accuracy\"] == 0.894 and abs(card[\"accuracy\"] - 0.944) <= 0.006\n    assert abs(card[\"auc\"] - 0.912) < 0.01 and abs(card[\"cv_accuracy\"][0] - 0.962) < 0.01\n    d = card[\"default\"]\n    assert d[\"threshold\"] == 0.5 and sum(d[\"confusion\"].values()) == 180\n    assert d[\"cost\"] == d[\"confusion\"][\"fp\"] * 1 + d[\"confusion\"][\"fn\"] * 10\n\ndef test_tuning():\n    \"the threshold follows the costs\"\n    X_train, X_test, y_train, y_test = _data()\n    card = report_card(LogisticRegression(max_iter=1000), X_train, y_train, X_test, y_test)\n    assert card[\"tuned\"][\"threshold\"] < 0.5, \"misses cost more, so flag more\"\n    assert card[\"tuned\"][\"recall\"] >= card[\"default\"][\"recall\"] and card[\"tuned\"][\"cost\"] <= card[\"default\"][\"cost\"]\n    cautious = report_card(LogisticRegression(max_iter=1000), X_train, y_train, X_test, y_test, cost_fp=10, cost_fn=1)\n    assert cautious[\"tuned\"][\"threshold\"] > 0.5 and cautious[\"tuned\"][\"precision\"] >= card[\"default\"][\"precision\"]\n    one = report_card(LogisticRegression(max_iter=1000), X_train, y_train, X_test, y_test, thresholds=[0.3])\n    assert one[\"tuned\"][\"threshold\"] == 0.3")

## Warm-up

Ninety per cent accurate, and useless. When one class is rare, **accuracy** rewards ignoring it. This is the normal situation, not an odd one: fraud, faults, disease, spam in a well-filtered inbox. Always ask how good the **baseline** is, and look at the kinds of mistake, not just their number.

## What accuracy hides

For a yes/no classifier, call the class you're looking for **positive** (faulty) and the other **negative** (fine). Every prediction lands in one of four boxes, the **confusion matrix**:

| | predicted positive | predicted negative |
| --- | --- | --- |
| **actually positive** | true positive (TP): caught | false negative (FN): **missed** |
| **actually negative** | false positive (FP): **false alarm** | true negative (TN): correctly passed |

Accuracy is `(TP + TN) / total`: it adds the boxes together and loses the difference between a missed fault and a false alarm, which usually cost very different amounts. scikit-learn's `confusion_matrix(y, pred)` returns the matrix with rows for the actual classes and columns for the predictions, negatives first: `[[TN, FP], [FN, TP]]`.

## Precision and recall

Two questions about the positives:

- **Precision** `= TP / (TP + FP)`: of the parts the model flagged, how many were really faulty? Low precision means many **false alarms**.
- **Recall** `= TP / (TP + FN)`: of the faulty parts, how many did the model catch? Low recall means many **misses**. (Also called sensitivity, or the true positive rate.)
- **F1** `= 2 × precision × recall / (precision + recall)` combines them; it's high only when both are.

Which matters more depends on what each mistake **costs**. A missed fault that reaches a customer costs far more than re-checking a good part, so the factory wants high recall. A spam filter that hides a real email does more harm than letting a spam through, so it wants high precision. Write the costs down before choosing a model.

Passing everything has a recall of 0: the number accuracy hid.

## Thresholds and ROC AUC

Most classifiers produce a **score** (with `predict_proba`, a probability) and `predict` turns it into a class at a **threshold** of 0.5. That's a choice, not a law. Lowering the threshold flags more parts: recall goes up, precision usually goes down. Raising it does the reverse. Choose the threshold from the costs, on **training** or validation data, never by peeking at the test set.

To compare models without picking a threshold, plot recall (the true positive rate) against the false positive rate `FP / (FP + TN)` for every threshold: the **ROC curve**. The area under it, **ROC AUC**, is 1 for a perfect ranking and 0.5 for random guessing. It has a plain meaning: the probability that a randomly chosen positive gets a **higher score** than a randomly chosen negative. On very imbalanced data, also look at the precision-recall trade-off directly, because a high AUC can still hide poor precision.

## Regression errors

For numeric predictions: **MAE** (mean absolute error) is in the label's units and easy to explain ("about £35 out"); **RMSE** (the square root of the MSE) is also in those units but weighs big errors more, so it's larger than MAE when some errors are large; **R²** compares the model with predicting the mean (1 is perfect, 0 is no better, negative is worse). Report a baseline next to each, and look at the **residuals** (`y − prediction`): the largest ones often reveal data errors or a kind of example the model doesn't understand.

One flat is £250 out, which pulls RMSE well above MAE: worth a look at what's special about it.

## Cross-validation and tuning

A single train/test split is one sample: a different split gives a different score. **k-fold cross-validation** splits the **training** data into `k` folds, trains on `k − 1` of them and scores on the remaining one, `k` times, so every example is scored once; report the **mean and spread** of the `k` scores. `cross_val_score(model, X, y, cv=5)` does it; for classification it uses **stratified** folds (each with the same class balance). Put preprocessing inside a pipeline so each fold fits its own scaler: otherwise the scaler sees the fold it's scored on, which is a leak.

Cross-validation is also how to choose **hyperparameters** (`k` for nearest neighbours, `max_depth` for a tree) honestly: **`GridSearchCV(model, {"n_neighbors": [1, 3, 5, 7]}, cv=5)`** cross-validates every setting on the training data and refits the best. The test set is used **once**, at the very end, to report the final score. If you tune by looking at test scores, the test score stops being an honest estimate: you've fitted to it.

## Practice

## Quiz

## Daily challenge

## Before you go

For a classifier you might build (a spam filter, a fault detector, a "will this bus be late" predictor), write down what a false positive and a false negative would each cost, in money, time or harm. Which would you rather have more of, and so which of precision and recall would you favour? Tomorrow is **a neural network from scratch**: the model behind most of modern machine learning, built from NumPy and the chain rule.